In [1]:
import duckdb                                                                                                                                  
import os
from dotenv import load_dotenv                                                                                                                 
                                                    
load_dotenv()                                                                                                                                  
con = duckdb.connect("md:warehouse_dev")

In [2]:
con.execute("""
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
  AND table_schema IN ('bronze', 'silver')
ORDER BY table_schema, table_name;
""").df()

,table_schema,table_name


In [3]:
gen_ex_df = con.execute("SELECT * FROM bronze.generation_and_exchange").df()
gen_ex_df.shape

CatalogException: Catalog Error: Table with name generation_and_exchange does not exist!
Did you mean "information_schema.constraint_table_usage"?

LINE 1: SELECT * FROM bronze.generation_and_exchange
                      ^

In [ ]:
gen_ex_df.head()

,TimeUTC,TimeDK,PriceArea,Version,GrossConsumptionMWh,CO2PerkWh,_loaded_at
0,2025-04-25 23:00:00+02:00,2025-04-25 23:00:00,DK1,Preliminary,2165.789065,175.729410,2026-04-26 12:45:26.703715+02:00
1,2025-04-25 23:00:00+02:00,2025-04-25 23:00:00,DK2,Preliminary,1335.228397,58.198583,2026-04-26 12:45:26.703715+02:00
2,2025-04-25 22:00:00+02:00,2025-04-25 22:00:00,DK1,Preliminary,2219.739227,207.237636,2026-04-26 12:45:26.703715+02:00
3,2025-04-25 22:00:00+02:00,2025-04-25 22:00:00,DK2,Preliminary,1397.627298,53.386836,2026-04-26 12:45:26.703715+02:00
4,2025-04-25 21:00:00+02:00,2025-04-25 21:00:00,DK1,Preliminary,2299.588722,210.520265,2026-04-26 12:45:26.703715+02:00


In [ ]:
cons_df = con.execute("SELECT * FROM silver.gross_consumption").df().sort_values("TimeUTC", ascending=False)
cons_df.shape

(58126, 5)

In [ ]:
cons_df.head()

,TimeUTC,TimeDK,PriceArea,Version,GrossConsumptionMWh
58125,2025-04-25 23:00:00+02:00,2025-04-25 23:00:00,DK2,Preliminary,1335.228397
58124,2025-04-25 23:00:00+02:00,2025-04-25 23:00:00,DK1,Preliminary,2165.789065
58123,2025-04-25 22:00:00+02:00,2025-04-25 22:00:00,DK2,Preliminary,1397.627298
58122,2025-04-25 22:00:00+02:00,2025-04-25 22:00:00,DK1,Preliminary,2219.739227
58121,2025-04-25 21:00:00+02:00,2025-04-25 21:00:00,DK2,Preliminary,1490.197105


In [ ]:
cons_df.Version.unique()

<StringArray>
['Preliminary', 'Final']
Length: 2, dtype: str

In [ ]:
import plotly.express as px

px.line(cons_df, x='TimeDK', y='GrossConsumptionMWh', color='PriceArea', title='Gross Consumption MWh over Time')